# Tour Recommender Training (Synthetic Dataset)
Train a personalized tour recommendation classifier from `training_interactions.csv`.


## 1. Upload Dataset
Upload `training_interactions.csv` from `datasets/synthetic_tour_recommendation/` to Colab, or mount Google Drive and update `DATA_PATH`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib

DATA_PATH = 'training_interactions.csv'
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
features = [
    'tag_similarity',
    'destination_match',
    'price_match',
    'duration_match',
    'price',
    'duration',
    'avg_rating',
    'review_count',
    'budget_max',
    'preferred_duration',
]
target = 'label'

X = df[features].copy()
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=4,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1,
)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print(classification_report(y_test, pred))
print('ROC AUC:', roc_auc_score(y_test, proba))

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
importances

In [ ]:
artifact = {
    'model': model,
    'features': features,
    'threshold': 0.5,
}
joblib.dump(artifact, 'tour_recommender_model.pkl')
print('Saved tour_recommender_model.pkl')

## Optional: XGBoost
If Colab has xgboost installed, you can train a stronger classifier.

In [ ]:
# Optional
# !pip install xgboost
# from xgboost import XGBClassifier
# xgb = XGBClassifier(
#     n_estimators=300, max_depth=5, learning_rate=0.05,
#     subsample=0.9, colsample_bytree=0.9, eval_metric='logloss', random_state=42
# )
# xgb.fit(X_train, y_train)
# xgb_proba = xgb.predict_proba(X_test)[:, 1]
# print('XGB ROC AUC:', roc_auc_score(y_test, xgb_proba))
# joblib.dump({'model': xgb, 'features': features, 'threshold': 0.5}, 'tour_recommender_xgb.pkl')